## SPARKS TOOLSET
    -Spark is composed of these primitives—​the lower-level APIs and the Structured APIs—​and then a series of standard libraries for additional functionality.
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781491912201/files/assets/spdg_0101.png" width="500">

## RUNNING PRODUCTION APPLICATIONS
    -Spark makes it easy to develop and create big data programs.
    -Spark-submit does one thing: it lets you send your application code to a cluster and launch it to execute there.
    -You can do this with all of Spark’s support cluster managers including Standalone, Mesos, and YARN.

## STRUCTURED STREAMING
    -Structured Streaming is a high-level API for stream processing that became production-ready in Spark 2.2.
    -The best thing about Structured Streaming is that it allows you to rapidly and quickly extract value out of streaming systems with virtually no code changes. 

In [1]:
# First we will create a dataframe and alaso create a schema from this static dataset
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Streaming').getOrCreate();

26/03/30 15:06:51 WARN Utils: Your hostname, intern7 resolves to a loopback address: 127.0.0.1; using 10.10.42.90 instead (on interface enp2s0)
26/03/30 15:06:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 15:06:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


    -Sale hours during which a given customer (identified by CustomerId) makes a large purchase. For example, let’s add a total cost column and see on what days a customer spent the most.

In [4]:
staticDataFrame=spark.read.format('csv').load('file:///home/unik/OReily/by-day/*csv',header=True,inferSchema=True);
staticDataFrame.createOrReplaceTempView('retail_data')
staticSchema=staticDataFrame.schema

In [8]:
from pyspark.sql.functions import window, column, desc, col
staticDataFrame\
  .selectExpr(
    "CustomerId",
    "(UnitPrice * Quantity) as total_cost",
    "InvoiceDate")\
  .groupBy(col("CustomerId"), window(col("InvoiceDate"), "1 day"))\
  .sum("total_cost")\
  .show();

[Stage 16:===========>                                             (2 + 8) / 10]

+----------+--------------------+------------------+
|CustomerId|              window|   sum(total_cost)|
+----------+--------------------+------------------+
|   16057.0|{2011-12-05 05:45...|             -37.6|
|   14126.0|{2011-11-29 05:45...| 643.6300000000001|
|   13500.0|{2011-11-16 05:45...| 497.9700000000001|
|   17160.0|{2011-11-08 05:45...| 516.8499999999999|
|   15608.0|{2011-11-11 05:45...|             122.4|
|   15253.0|{2011-11-23 05:45...|             277.6|
|   15124.0|{2011-11-17 05:45...|             93.44|
|   12539.0|{2011-11-17 05:45...|           1050.66|
|   13658.0|{2011-11-30 05:45...| 542.4000000000001|
|   17396.0|{2011-10-31 05:45...|             495.0|
|   13576.0|{2011-11-10 05:45...| 543.3600000000001|
|   15111.0|{2011-11-10 05:45...|329.67999999999995|
|   17419.0|{2011-10-06 05:45...|465.54999999999995|
|   15749.0|{2011-04-18 05:45...|-1462.500000000001|
|   15769.0|{2011-04-18 05:45...|122.03999999999999|
|   18219.0|{2011-04-18 05:45...|            2

In [11]:
streamingDataFrame = spark.readStream\
    .schema(staticSchema)\
    .option("maxFilesPerTrigger", 1)\
    .format("csv")\
    .option("header", "true")\
    .load("file:///home/unik/OReily/by-day/*csv")
streamingDataFrame.isStreaming

True

## MACHINE LEARNING AND ADVANCED ANALYTICS

In [12]:
staticDataFrame.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



    -Machine learning algorithms in MLlib require that data is represented as numerical values. Our current data is represented by a variety of different types, including timestamps, integers, and strings. Therefore we need to transform this data into some numerical representation. In this instance, we’ll use several DataFrame transformations to manipulate our date data: